In [1]:
import pandas as pd
import numpy as np
from datetime import datetime

from setting_for_sda.path_setting import path_list
from setting_for_sda.date_setting import Date_Setting


import lib.stats.stats as st
from lib.utils.file_io import *
from lib.utils.statistics import *
from lib.utils.settings import set_matplotlib
from lib.utils.datetime_handler import calc_rel_period
from lib.visualization.distribution_collector import (calc_top_mid_bottom_tags_prop, compute_proportion_period)
from lib.visualization.plot_generator import PlotGen
from matplotlib import pyplot as plt

import matplotlib as mpl
import lib.visualization.font_setting as font_setting
mpl.rcParams['font.family'] = font_setting.init_font()

import matplotlib.pyplot as plt
import statsmodels.formula.api as smf
from scipy.stats import spearmanr



Helvetica /home/mghan/.fonts/Helvetica/Helvetica Oblique.ttf
Registered font name: Helvetica


In [7]:
def gini_coefficient(values):
    """
    Gini coefficient 계산 (unordered 정의)
    G = sum_i sum_j |x_i - x_j| / (2n sum x_i)
    효율을 위해 정렬된 형태로 계산.
    
    Parameters
    ----------
    values : array-like
        태그별 proportion (혹은 count) — 합이 1일 필요 없음
    
    Returns
    -------
    float
        Gini coefficient in [0, 1]
    """
    x = np.asarray(values, dtype=float)
    x = x[x >= 0]
    if x.sum() == 0 or len(x) < 2:
        return 0.0
    # 표준 공식: 정렬 후 계산
    x_sorted = np.sort(x)
    n = len(x_sorted)
    cumx = np.cumsum(x_sorted)
    # G = (2 * sum(i * x_i) - (n+1) * sum(x)) / (n * sum(x))
    index = np.arange(1, n + 1)
    return (2.0 * np.sum(index * x_sorted) - (n + 1) * cumx[-1]) / (n * cumx[-1])

def compute_N_by_week(df, period_col='rel_week', weight_col='tot_pct'):
    """주차별 질문 수 (tot_pct를 질문 수로 쓰는 경우)"""
    N = df.groupby(period_col)[weight_col].sum()
    try:
        N.index = N.index.astype(int)
    except (ValueError, TypeError):
        pass
    return N.sort_index()


# ---------------- 지표 3: N-adjusted Gini ----------------
def n_adjusted_metric(metric_series, N_series, cutoff=0):
    """
    지표 ~ log(N) + week + post + week:post 회귀.
    N 효과 통제 후 시간/ChatGPT 효과 추정.
    
    Returns
    -------
    model: fitted OLS model
    adjusted: N 효과 제거한 잔차 시계열
    summary: DID coefficient 등 요약
    """
    common = metric_series.index.intersection(N_series.index)
    df = pd.DataFrame({
        'value': metric_series.loc[common].values,
        'logN': np.log(N_series.loc[common].values),
        'week': np.array(common, dtype=float),
    })
    df['post'] = (df['week'] >= cutoff).astype(int)

    # Full 모델
    model = smf.ols('value ~ logN + week + post + week:post', data=df).fit()

    # N 효과만 제거한 잔차 (시간, post, 상호작용 효과는 남김)
    # predict with N at mean level
    mean_logN = df['logN'].mean()
    df_counterfactual = df.copy()
    df_counterfactual['logN'] = mean_logN
    predicted_without_N_variation = model.predict(df_counterfactual)
    adjusted = pd.Series(predicted_without_N_variation.values, index=df['week'].values.astype(int))

    summary = {
        'logN_coef': model.params['logN'],
        'logN_pval': model.pvalues['logN'],
        'week_coef': model.params['week'],
        'week_pval': model.pvalues['week'],
        'post_coef': model.params['post'],
        'post_pval': model.pvalues['post'],
        'did_coef': model.params['week:post'],
        'did_pval': model.pvalues['week:post'],
        'r_squared': model.rsquared,
    }
    return model, adjusted, summary


# ---------------- N 의존성 진단 ----------------
def diagnose_n_dependency(metric_series, N_series, name='metric'):
    """Raw 상관 + Detrended 상관"""
    common = metric_series.index.intersection(N_series.index)
    weeks = np.array(common, dtype=float)
    y = metric_series.loc[common].values
    logN = np.log(N_series.loc[common].values)

    # Raw
    raw_r, raw_p = spearmanr(logN, y)

    # Detrended
    import statsmodels.api as sm
    def detrend(v):
        X = sm.add_constant(weeks)
        return sm.OLS(v, X).fit().resid
    y_resid = detrend(y)
    N_resid = detrend(logN)
    det_r, det_p = spearmanr(N_resid, y_resid)

    return {
        'metric': name,
        'raw_corr': raw_r, 'raw_p': raw_p,
        'detrended_corr': det_r, 'detrended_p': det_p,
    }


def plot_panelC(raw_gini_series,adjusted_gini_series,
                figsize=(12, 6), save_path=None, lang = 'python',smooth_window=4):
    """
    4-panel 그림:
    (A) 질문 수 N 시계열 — 감소 추세 확인
    (B) Baseline Top-30 share — 주류 태그 밀려남
    (C) Long-tail share — 롱테일 부상
    (D) Gini: raw vs N-adjusted — N 통제해도 변화 있음
    """
    fig, axes = plt.subplots(1, 1, figsize=figsize, sharex=True)
    
    idx = raw_gini_series.index.values
    axes.plot(idx, raw_gini_series.values, linewidth=1.2,
                 color='#6A4C93', alpha=0.5, label='Raw Gini')
    if adjusted_gini_series is not None:
        idx_adj = adjusted_gini_series.index.values
        axes.plot(idx_adj, adjusted_gini_series.values,
                     linewidth=2.2, color='#C73E1D',
                     label='N-adjusted Gini (predicted at mean log N)')
    axes.set_ylabel('Gini coefficient', fontsize=11)
    axes.set_title('(D) Gini: N-adjusted trajectory shows residual change',
                      fontsize=11, loc='left')
    axes.grid(alpha=0.3)
    axes.legend(loc='best', fontsize=9)

    fig.suptitle('Evidence for topic diversification despite declining volume',
                 fontsize=13, y=0.995, fontweight='bold')
    fig.tight_layout()
    if save_path:
        fig.savefig(save_path, dpi=150, bbox_inches='tight')
    plt.close()




In [8]:
for idx, lang in enumerate(CONSTANTS.languages_from2020to2022) :
    plotgen = PlotGen(idx, 'tag')
    print(f'[Start....] drawing figures for {lang} language')


    df = load_df(plotgen.data_dir, ['cdate' , 'id' , 'tag', 'cnt', 'tot_cnt', 'pct'])
    df = calc_rel_period(df, plotgen.std_date, date_col = 'cdate', period = 7)
    df_proportion = compute_proportion_period(df,period = 'rel_week', type = 'tag', value_col='pct')
    N_series = compute_N_by_week(df_proportion, weight_col='pct')

    # 지표 3: Raw Gini
    gini_rows = {}
    for w, grp in df_proportion.groupby('rel_week'):
        gini_rows[w] = gini_coefficient(grp['proportion'].values)
    raw_gini = pd.Series(gini_rows).sort_index()
    try:
        raw_gini.index = raw_gini.index.astype(int)
    except (ValueError, TypeError):
        pass

    # N-adjusted
    model, adjusted_gini, summary = n_adjusted_metric(raw_gini, N_series, cutoff=0)
    print("\n=== N-adjusted Gini regression ===")
    for k, v in summary.items():
        print(f"  {k}: {v:.6f}" if isinstance(v, float) else f"  {k}: {v}")  

    # 종합 시각화
    plot_panelC(raw_gini, adjusted_gini, 
                             save_path=f'./fig/{idx}_PanelC_for_{lang}_gini.png', lang = lang)


    print("\nSaved figures.")

[Start....] drawing figures for python language

=== N-adjusted Gini regression ===
  logN_coef: 0.066685
  logN_pval: 0.000000
  week_coef: -0.000122
  week_pval: 0.000137
  post_coef: -0.012260
  post_pval: 0.000001
  did_coef: -0.000693
  did_pval: 0.000000
  r_squared: 0.994194

Saved figures.
[Start....] drawing figures for javascript language

=== N-adjusted Gini regression ===
  logN_coef: 0.059857
  logN_pval: 0.000000
  week_coef: -0.000073
  week_pval: 0.082022
  post_coef: -0.000853
  post_pval: 0.785373
  did_coef: -0.000884
  did_pval: 0.000000
  r_squared: 0.990917

Saved figures.
[Start....] drawing figures for java language

=== N-adjusted Gini regression ===
  logN_coef: 0.067649
  logN_pval: 0.000000
  week_coef: -0.000082
  week_pval: 0.066838
  post_coef: -0.006128
  post_pval: 0.057000
  did_coef: -0.000236
  did_pval: 0.001877
  r_squared: 0.981129

Saved figures.
[Start....] drawing figures for c# language

=== N-adjusted Gini regression ===
  logN_coef: 0.098981

In [9]:
plotgen = PlotGen(0, 'tag')
# print(f'[Start....] drawing figures for {lang} language')


df = load_df(plotgen.data_dir, ['cdate' , 'id' , 'tag', 'cnt', 'tot_cnt', 'pct'])
df = calc_rel_period(df, plotgen.std_date, date_col = 'cdate', period = 7)
df_proportion = compute_proportion_period(df,period = 'rel_week', type = 'tag', value_col='pct')
N_series = compute_N_by_week(df_proportion, weight_col='pct')

In [10]:
N_series

rel_week
-104    5037.0
-103    5002.0
-102    4692.0
-101    3976.0
-100    4148.0
         ...  
 151     139.0
 152     135.0
 153     140.0
 154     133.0
 155     114.0
Name: pct, Length: 260, dtype: float64